In [ ]:
# 0) Colab runtime setup + git pull
import os
from pathlib import Path
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

REPO_URL = 'https://github.com/Swapperss/FinanceQAAssistance.git'
PROJECT_ROOT = Path('/content/FinanceQAAssistance')

if not PROJECT_ROOT.exists():
    !git clone {REPO_URL} {PROJECT_ROOT}
else:
    print('Repository exists, pulling latest changes...')
    %cd {PROJECT_ROOT}
    !git pull

%cd {PROJECT_ROOT}
!git branch --show-current

!pip -q install -U pip
!pip -q install -r requirements.txt
!pip -q install -U transformers accelerate peft bitsandbytes datasets huggingface_hub

print('Project root ready:', PROJECT_ROOT)

In [ ]:
# 1) Finance instruction dataset setup and generation
import json
import re
from pathlib import Path
import sys
import pandas as pd

current_path = Path.cwd().resolve()
project_root = None
for candidate_path in [current_path, *current_path.parents]:
    if (candidate_path / 'config' / 'config.py').exists():
        project_root = candidate_path
        break
if project_root is None:
    raise FileNotFoundError('Could not locate config/config.py from the current working directory.')
project_root = Path(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import importlib.util
config_spec = importlib.util.spec_from_file_location('finance_config', project_root / 'config' / 'config.py')
config_module = importlib.util.module_from_spec(config_spec)
assert config_spec.loader is not None
config_spec.loader.exec_module(config_module)
Config = config_module.Config
config = Config()

raw_dir = project_root / config.raw_data_dir
instruction_dir = project_root / config.instruction_dir
stage1_merged_model_dir = project_root / config.stage1_merged_model_dir
instruction_output_dir = project_root / config.instruction_output_dir
instruction_adapter_dir = project_root / config.instruction_adapter_dir
instruction_dataset_path = instruction_dir / config.instruction_dataset_filename

for folder_path in [instruction_dir, stage1_merged_model_dir, instruction_output_dir, instruction_adapter_dir]:
    folder_path.mkdir(parents=True, exist_ok=True)

raw_csv_path = raw_dir / config.preferred_raw_csv_filename
if not raw_csv_path.exists():
    matches = sorted(raw_dir.glob(config.raw_csv_pattern))
    if not matches:
        raise FileNotFoundError(f'No complaints CSV found in {raw_dir}')
    raw_csv_path = matches[-1]

df = pd.read_csv(raw_csv_path, dtype=str, low_memory=False).fillna('')

def normalize_text(value):
    return re.sub(r'\s+', ' ', str(value)).strip()

def finance_answer_from_issue(issue_text):
    issue_text = issue_text.lower()

    if any(keyword in issue_text for keyword in ['unauthorized', 'not authorized', 'fraud', 'identity theft']):
        return 'Contact your bank or card issuer immediately, lock or replace the card if needed, dispute the charge in writing, and monitor the account for any further suspicious activity.'

    if any(keyword in issue_text for keyword in ['fees or interest', 'incorrect fees or interest', 'charged too much interest', 'annual fee', 'fee']):
        return 'Review the statement, contact the bank or card issuer, dispute the fee in writing, and request a reversal or correction if the charge looks incorrect.'

    if any(keyword in issue_text for keyword in ['payment', 'late', 'posted late', 'not posted']):
        return 'Provide proof of payment, ask the bank to review the posting timeline, and request removal of any late fees or interest caused by a bank-side delay.'

    if any(keyword in issue_text for keyword in ['system transition', 'migration']):
        return 'Ask for a manual review, explain the system transition issue, and request reversal of any fees or interest caused by the transition.'

    return 'Contact the bank customer service team, explain the issue in writing, ask for a manual review, and keep copies of all statements and correspondence.'

def build_instruction_record(row):
    product = normalize_text(row.get('Product', ''))
    issue = normalize_text(row.get('Issue', ''))
    sub_issue = normalize_text(row.get('Sub-issue', ''))
    complaint_text = normalize_text(row.get(config.source_text_column, ''))

    if issue:
        instruction = f'What should I do if my complaint is about {issue.lower()}?'
    elif product:
        instruction = f'What should I do if I have a problem with my {product.lower()} account?'
    else:
        instruction = 'What should I do about this finance complaint?'

    input_parts = []
    if product:
        input_parts.append(f'Product: {product}')
    if issue:
        input_parts.append(f'Issue: {issue}')
    if sub_issue:
        input_parts.append(f'Sub-issue: {sub_issue}')
    if complaint_text:
        input_parts.append(f'Complaint: {complaint_text[:1200]}')

    input_text = '\n'.join(input_parts).strip()
    output_text = finance_answer_from_issue(f'{product} {issue} {sub_issue}')

    return {
        'instruction': instruction,
        'input': input_text,
        'output': output_text,
    }

instruction_records = []
max_instruction_rows = min(400, len(df))
for _, row in df.head(max_instruction_rows).iterrows():
    record = build_instruction_record(row)
    if record['instruction'] and record['output']:
        instruction_records.append(record)

if not instruction_records:
    raise ValueError('No finance instruction records could be created from the CSV.')

with open(instruction_dataset_path, 'w', encoding='utf-8') as file_handle:
    for record in instruction_records:
        file_handle.write(json.dumps(record, ensure_ascii=False) + '\n')

print('Created instruction records:', len(instruction_records))
print('Instruction dataset saved to:', instruction_dataset_path)
print('Stage-1 merged model dir:', stage1_merged_model_dir)

In [ ]:
# 2) Load and tokenize instruction dataset
from datasets import load_dataset
from transformers import AutoTokenizer

instruction_dataset = load_dataset('json', data_files=str(instruction_dataset_path), split='train')

try:
    tokenizer = AutoTokenizer.from_pretrained(config.non_instruction_adapter_repo_id, use_fast=True)
    print('Tokenizer loaded from continuation adapter repo.')
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)
    print('Tokenizer loaded from base model.')

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_instruction_record(record):
    instruction = str(record.get('instruction', '')).strip()
    input_text = str(record.get('input', '')).strip()
    output_text = str(record.get('output', '')).strip()

    if input_text:
        text = (
            f'### Instruction:\n{instruction}\n\n'
            f'### Input:\n{input_text}\n\n'
            f'### Response:\n{output_text}'
        )
    else:
        text = (
            f'### Instruction:\n{instruction}\n\n'
            f'### Response:\n{output_text}'
        )

    return {'text': text}

instruction_dataset = instruction_dataset.map(format_instruction_record)
instruction_datasets = instruction_dataset.train_test_split(test_size=config.test_size, seed=config.seed)
instruction_datasets['validation'] = instruction_datasets.pop('test')

response_prefix_ids = tokenizer('### Response:\n', add_special_tokens=False)['input_ids']

def find_subsequence(sequence, pattern):
    if not pattern:
        return -1
    last_start = len(sequence) - len(pattern)
    for start_index in range(last_start + 1):
        if sequence[start_index:start_index + len(pattern)] == pattern:
            return start_index
    return -1

def tokenize_instruction_function(examples):
    tokens = tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=config.block_size,
    )

    labels = []
    for input_ids, attention_mask in zip(tokens['input_ids'], tokens['attention_mask']):
        row_labels = list(input_ids)

        for idx, mask_value in enumerate(attention_mask):
            if mask_value == 0:
                row_labels[idx] = -100

        if config.instruction_train_response_only:
            response_start = find_subsequence(input_ids, response_prefix_ids)
            if response_start >= 0:
                for idx in range(response_start):
                    row_labels[idx] = -100

        labels.append(row_labels)

    tokens['labels'] = labels
    return tokens

instruction_tokenized_datasets = instruction_datasets.map(
    tokenize_instruction_function,
    batched=True,
    remove_columns=instruction_datasets['train'].column_names,
    desc='Tokenizing instruction dataset',
)

poc_train_rows = min(config.instruction_poc_train_rows, len(instruction_tokenized_datasets['train']))
poc_val_rows = min(config.instruction_poc_validation_rows, len(instruction_tokenized_datasets['validation']))
instruction_tokenized_datasets['train'] = instruction_tokenized_datasets['train'].select(range(poc_train_rows))
instruction_tokenized_datasets['validation'] = instruction_tokenized_datasets['validation'].select(range(poc_val_rows))

print('POC train rows:', len(instruction_tokenized_datasets['train']))
print('POC validation rows:', len(instruction_tokenized_datasets['validation']))

In [ ]:
# 3) Optional Hugging Face login for private continuation adapter
import os
from huggingface_hub import login

hf_read_token = os.getenv('HF_READ_TOKEN', '').strip() or os.getenv('HF_TOKEN', '').strip()
if hf_read_token:
    login(token=hf_read_token)
    print('Logged into Hugging Face using HF_READ_TOKEN/HF_TOKEN')
else:
    print('No HF_READ_TOKEN/HF_TOKEN found. If adapter is private, run notebook_login().')

In [ ]:
# 4) E2E style: load Stage-1 adapter, merge into base, then add NEW LoRA for instruction tuning
import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, PeftModel, TaskType, get_peft_model, prepare_model_for_kbit_training

use_cuda = torch.cuda.is_available()
print('CUDA available:', use_cuda)
print('Stage-1 adapter repo:', config.non_instruction_adapter_repo_id)

gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

# Load base model in full precision for reliable merge.
merge_dtype = torch.float16 if use_cuda else torch.float32
stage1_base_model_for_merge = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    torch_dtype=merge_dtype,
    trust_remote_code=True,
)

# Load non-instruction adapter and merge it into base weights.
stage1_model_with_adapter = PeftModel.from_pretrained(
    stage1_base_model_for_merge,
    config.non_instruction_adapter_repo_id,
)
merged_stage1_model = stage1_model_with_adapter.merge_and_unload()
merged_stage1_model.save_pretrained(str(stage1_merged_model_dir))
print('Merged Stage-1 model saved to:', stage1_merged_model_dir)

# Reload merged model in QLoRA mode for efficient instruction tuning.
if use_cuda:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        str(stage1_merged_model_dir),
        quantization_config=quantization_config,
        device_map='auto',
        trust_remote_code=True,
    )
    instruction_base_model = prepare_model_for_kbit_training(instruction_base_model)
else:
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        str(stage1_merged_model_dir),
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

instruction_base_model.config.use_cache = False

# Add a NEW LoRA adapter for the instruction stage (E2E parity behavior).
instruction_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

instruction_model = get_peft_model(instruction_base_model, instruction_lora_config)
instruction_model.print_trainable_parameters()

instruction_data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

instruction_training_args = TrainingArguments(
    output_dir=str(instruction_output_dir),
    num_train_epochs=config.instruction_num_train_epochs,
    max_steps=config.instruction_max_steps,
    per_device_train_batch_size=config.instruction_per_device_train_batch_size,
    per_device_eval_batch_size=config.instruction_per_device_eval_batch_size,
    gradient_accumulation_steps=config.instruction_gradient_accumulation_steps,
    learning_rate=config.instruction_learning_rate,
    warmup_ratio=config.instruction_warmup_ratio,
    weight_decay=config.instruction_weight_decay,
    logging_steps=config.instruction_logging_steps,
    logging_first_step=True,
    save_steps=config.instruction_save_steps,
    save_total_limit=config.instruction_save_total_limit,
    eval_strategy='no',
    fp16=use_cuda,
    bf16=False,
    report_to='none',
    remove_unused_columns=False,
)

instruction_trainer = Trainer(
    model=instruction_model,
    args=instruction_training_args,
    train_dataset=instruction_tokenized_datasets['train'],
    eval_dataset=instruction_tokenized_datasets['validation'],
    data_collator=instruction_data_collator,
)

print('Instruction Trainer is ready (E2E merge + new LoRA mode).')

In [ ]:
# 5) Train, save adapter, and push to Hugging Face Hub
from huggingface_hub import create_repo, login

instruction_train_result = instruction_trainer.train()
print('Instruction fine-tuning completed.')
print(instruction_train_result)

instruction_trainer.model.save_pretrained(str(instruction_adapter_dir))
tokenizer.save_pretrained(str(instruction_adapter_dir))
print('Instruction adapter saved to:', instruction_adapter_dir)

write_token = os.getenv('HF_WRITE_TOKEN', '').strip() or os.getenv('HF_TOKEN', '').strip()
if write_token:
    login(token=write_token)
else:
    print('No HF_WRITE_TOKEN/HF_TOKEN found. If repo is private, run notebook_login() before push.')

instruction_repo_id = config.instruction_adapter_repo_id
create_repo(repo_id=instruction_repo_id, repo_type='model', private=True, exist_ok=True)
instruction_trainer.model.push_to_hub(instruction_repo_id, private=True)
tokenizer.push_to_hub(instruction_repo_id, private=True)
print('Pushed instruction adapter to:', instruction_repo_id)

In [ ]:
# 6) Reload instruction adapter for inference (from HF Hub)
from peft import PeftModel

if use_cuda:
    inference_instruction_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map='auto',
        trust_remote_code=True,
    )
else:
    inference_instruction_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

inference_instruction_model = PeftModel.from_pretrained(
    inference_instruction_base_model,
    config.instruction_adapter_repo_id,
)
inference_instruction_model.eval()
print('Instruction inference model ready from:', config.instruction_adapter_repo_id)

In [ ]:
# 7) Instruction inference helper and test
def build_instruction_prompt(instruction, input_text=''):
    instruction = instruction.strip()
    input_text = input_text.strip()

    if input_text:
        return (
            f'### Instruction:\n{instruction}\n\n'
            f'### Input:\n{input_text}\n\n'
            f'### Response:\n'
        )

    return (
        f'### Instruction:\n{instruction}\n\n'
        f'### Response:\n'
    )

def generate_instruction_response(instruction, input_text='', max_new_tokens=None):
    prompt = build_instruction_prompt(instruction, input_text)
    inputs = tokenizer(prompt, return_tensors='pt').to(inference_instruction_model.device)

    if max_new_tokens is None:
        max_new_tokens = config.instruction_inference_max_new_tokens

    with torch.no_grad():
        outputs = inference_instruction_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=config.instruction_inference_do_sample,
            temperature=config.instruction_inference_temperature,
            top_p=config.instruction_inference_top_p,
            repetition_penalty=config.instruction_inference_repetition_penalty,
            no_repeat_ngram_size=config.instruction_inference_no_repeat_ngram_size,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer_text = generated_text[len(prompt):].strip()
    return answer_text

instruction_test_questions = [
    'What should I do if my bank charged an incorrect fee?',
    'How can I dispute an unauthorized charge on my credit card?',
    'What can I do if my payment was posted late because of a bank error?',
    'How do I request a refund for an unfair interest charge?',
    'What should I do if my account was charged fees after a system transition?',
]

for index, question in enumerate(instruction_test_questions, start=1):
    response = generate_instruction_response(question)
    print(f'Q{index}:', question)
    print(f'A{index}:', response)
    print('-' * 80)